<a href="https://colab.research.google.com/github/wallafsilva03-spec/fieldops-piloto-/blob/claude%2Fagri-gps-colab-notebook-gb9sk8/FieldOps_Comit%C3%AA_LA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚜 FieldOps — Análise de Telemetria GPS por Área Agrícola

Este notebook do **Google Colab** identifica em qual área de um **KML** (fazendas,
talhões, setores ou blocos) cada ponto GPS de telemetria ocorreu e calcula
**quanto tempo cada equipamento trabalhou dentro de cada área**.

**Entradas:**
1. Um arquivo **KML** com polígonos das áreas agrícolas.
2. Um arquivo **ZIP** contendo um ou mais CSVs de telemetria com as colunas:
   `ceqid, nickname, vin, name, numeric_value, text_value, uom, event_timestamp, lat, lon`.

**Saída:**
- `resumo_por_equipamento.xlsx` — por **data / equipamento / área**: horas
  trabalhadas, **horas efetivas** (carga ≥ 30% + elevador `forward`), **horas com
  piloto ligado** (orientação automática `on`) e quantidade de pontos.

> **Como usar:** execute as células **na ordem**, de cima para baixo
> (`Runtime → Run all` ou `Ctrl/Cmd + Enter` célula a célula).

O código usa **GeoPandas Spatial Join**, **índice espacial** e **operações
vetorizadas**, com dtypes econômicos (`category`/`float32`) e liberação de
memória, suportando **milhões** de linhas de telemetria sem loops linha a linha.


## ⚙️ Etapa 1 — Instalação automática das dependências

Instala todas as bibliotecas necessárias. Compatível com o ambiente do Google
Colab. Pode levar 1–2 minutos na primeira execução.

> Se aparecer um aviso pedindo para **reiniciar o ambiente** ("Restart runtime"),
> reinicie e execute novamente a partir daqui.


In [1]:
# Instalacao silenciosa das dependencias no Google Colab.
# pyogrio + fiona dao suporte de leitura/escrita vetorial ao GeoPandas.
import sys, subprocess

PACOTES = [
    "pandas",
    "geopandas",
    "shapely",
    "fiona",
    "pyogrio",
    "fastkml",
    "openpyxl",
    "tqdm",
    "lxml",          # backend usado pelo fastkml para parsear o KML
]

print("Instalando dependencias (isso pode levar alguns minutos)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PACOTES],
    check=True,
)
print("Dependencias instaladas com sucesso.")


Instalando dependencias (isso pode levar alguns minutos)...
Dependencias instaladas com sucesso.


### Importações e configurações globais

Importa as bibliotecas e define algumas constantes usadas no restante do notebook.


In [2]:
import os
import re
import gc
import glob
import zipfile
import warnings
from io import BytesIO

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.validation import make_valid
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tqdm.pandas()

# ----------------------------- Parametros -----------------------------------
EPSG_GEO = 4326                 # WGS84 (lat/lon)
MAX_GAP_SECONDS = 10 * 60       # Intervalo maximo valido entre pontos = 10 minutos
COLUNAS_OBRIGATORIAS = ["ceqid", "lat", "lon", "event_timestamp"]

# Fuso horario local usado para DERIVAR A DATA do registro (coluna 'data')
# e exibir o horario local. Os timestamps sao lidos em UTC e convertidos
# para este fuso. Padrao: horario de Brasilia.
TIMEZONE_LOCAL = "America/Sao_Paulo"

# ============================================================================
# CONFIGURACAO DOS ESTADOS DE OPERACAO (a partir da coluna 'name')
# ----------------------------------------------------------------------------
# A telemetria vem em formato "longo": cada linha e UMA leitura de um sinal,
# identificado pela coluna 'name', com o valor em 'numeric_value' ou 'text_value'.
# Aqui definimos quais sinais representam cada estado. A correspondencia ignora
# acentos e maiusculas/minusculas (ex.: "Status da orientacao automatica").
#
# Se os nomes dos seus sinais forem diferentes, ajuste as palavras-chave abaixo.
# A Etapa 3.5 imprime os nomes de sinais encontrados para facilitar o ajuste.
# ============================================================================

# --- TEMPO EFETIVO = carga de motor >= 30% E status do elevador = "forward" ---
# Obs.: o VALOR da carga vem em 'numeric_value'; a unidade ('%') fica em 'uom'.
LIMIAR_CARGA_MOTOR = 30.0                       # % minima de carga do motor
SINAL_CARGA_MOTOR = ["carga", "motor"]          # 'name' deve conter TODAS estas palavras
SINAL_STATUS_ELEVADOR = ["elevador"]            # 'name' deve conter estas palavras
VALOR_ELEVADOR_EFETIVO = ["forward"]            # valor (text_value) que indica efetivo

# --- PILOTO LIGADO = "Status da orientacao automatica" = on ------------------
SINAL_PILOTO = ["orientacao", "automatica"]     # 'name' = Status da orientacao automatica
VALORES_LIGADO = ["on", "ligado", "ativo", "ativado", "engaged",
                  "true", "sim", "yes", "1", "habilitado"]

# Pasta de trabalho onde os arquivos serao salvos
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(WORKDIR)


# --------------------------- Funcoes auxiliares -----------------------------
def remover_acentos(texto):
    """Normaliza string: minuscula, sem acentos e sem espacos extras."""
    import unicodedata
    if texto is None:
        return ""
    s = str(texto)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.lower().strip()


def corrigir_mojibake(texto):
    """Tenta reparar acentuacao quebrada (mojibake) do tipo 'orientaÃ§Ã£o' -> 'orientação'.

    Ocorre quando um texto UTF-8 foi lido como latin-1/cp1252. So aplica a
    correcao quando ela claramente melhora o texto (remove os caracteres 'Ã/Â').
    """
    if not isinstance(texto, str):
        return texto
    if "Ã" not in texto and "Â" not in texto:
        return texto
    try:
        reparado = texto.encode("latin-1").decode("utf-8")
        return reparado
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto


def contem_todas(serie_norm, palavras):
    """Mascara booleana: True onde a serie normalizada contem TODAS as palavras."""
    mask = pd.Series(True, index=serie_norm.index)
    for p in palavras:
        mask &= serie_norm.str.contains(remover_acentos(p), na=False, regex=False)
    return mask


def para_numero(serie):
    """Converte texto -> numero, tolerando virgula como separador decimal (pt-BR).

    Ex.: '35,5' -> 35.5 ; '1.234,5' -> nao mexe (tem ponto). Valores nao numericos
    viram NaN. Essencial para a carga do motor vinda com virgula decimal.
    """
    s = serie.astype(str).str.strip()
    # troca virgula decimal por ponto apenas quando NAO houver ponto no valor
    tem_virgula = s.str.contains(",", na=False) & ~s.str.contains(r"\.", na=False, regex=True)
    s = s.mask(tem_virgula, s.str.replace(",", ".", regex=False))
    return pd.to_numeric(s, errors="coerce")


print("Versoes:")
print("  pandas    :", pd.__version__)
print("  geopandas :", gpd.__version__)
print("Diretorio de trabalho:", WORKDIR)


Versoes:
  pandas    : 2.2.2
  geopandas : 1.1.3
Diretorio de trabalho: /content


## 📤 Etapa 2 — Upload dos arquivos

Faça o upload manual dos dois arquivos pelo navegador:

- **KML** com os polígonos das áreas (`.kml`)
- **ZIP** com os CSVs de telemetria (`.zip`)

Você pode selecionar os dois de uma vez ou rodar a célula duas vezes.


In [3]:
# Upload manual via navegador. Funciona apenas no Google Colab.
try:
    from google.colab import files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
    print("Aviso: nao parece estar no Google Colab.")
    print("Coloque 'arquivo.kml' e 'arquivo.zip' no diretorio de trabalho manualmente.")

if EM_COLAB:
    print("Selecione o arquivo .KML e o arquivo .ZIP (pode selecionar os dois juntos):")
    enviados = files.upload()
    for nome in enviados:
        print(f"  recebido: {nome} ({len(enviados[nome])/1024:.1f} KB)")

# Localiza automaticamente os arquivos enviados no diretorio de trabalho
def _encontrar(extensao):
    arquivos = sorted(glob.glob(os.path.join(WORKDIR, f"*{extensao}")))
    return arquivos

kmls = _encontrar(".kml")
zips = _encontrar(".zip")

if not kmls:
    raise FileNotFoundError("Nenhum arquivo .kml encontrado. Faca o upload do KML.")
if not zips:
    raise FileNotFoundError("Nenhum arquivo .zip encontrado. Faca o upload do ZIP.")

KML_PATH = kmls[0]
ZIP_PATH = zips[0]
print("\nArquivos selecionados:")
print("  KML:", KML_PATH)
print("  ZIP:", ZIP_PATH)
if len(kmls) > 1:
    print("  (varios KMLs encontrados; usando o primeiro)")
if len(zips) > 1:
    print("  (varios ZIPs encontrados; usando o primeiro)")


Selecione o arquivo .KML e o arquivo .ZIP (pode selecionar os dois juntos):


Saving CEM.kml to CEM.kml
Saving wallaf789.zip to wallaf789.zip
  recebido: CEM.kml (62406.7 KB)
  recebido: wallaf789.zip (20919.7 KB)

Arquivos selecionados:
  KML: /content/CEM.kml
  ZIP: /content/wallaf789.zip


## 📖 Etapa 3 — Leitura dos dados

### 3.1 — Leitura do KML → GeoDataFrame

Lê o KML, extrai **todos os polígonos** e **todos os atributos** disponíveis
(nome, descrição, e quaisquer campos de *ExtendedData*). O notebook tenta
primeiro o leitor nativo do GeoPandas (`pyogrio`/`fiona`) e, se falhar, usa um
parser manual baseado em `fastkml`/`lxml`, garantindo robustez contra KMLs
com estruturas variadas.


In [4]:
def ler_kml_geopandas(caminho):
    """Tenta ler o KML usando os drivers nativos do GeoPandas (pyogrio/fiona).

    O driver KML do GDAL nem sempre ativa por padrao; habilitamos o suporte.
    Retorna um GeoDataFrame ou levanta excecao.
    """
    # Habilita o driver KML no fiona/GDAL (necessario em alguns ambientes)
    try:
        import fiona
        fiona.drvsupport.supported_drivers["KML"] = "rw"
        fiona.drvsupport.supported_drivers["LIBKML"] = "rw"
    except Exception:
        pass

    ultima_excecao = None
    # pyogrio costuma ler todas as camadas/pastas do KML de uma vez
    for engine in ("pyogrio", "fiona"):
        try:
            gdf = gpd.read_file(caminho, engine=engine)
            if len(gdf) > 0:
                return gdf
        except Exception as e:  # tenta o proximo engine
            ultima_excecao = e
    if ultima_excecao:
        raise ultima_excecao
    raise ValueError("KML lido, porem sem feicoes.")


def ler_kml_fastkml(caminho):
    """Parser manual de KML usando lxml diretamente.

    Percorre todos os <Placemark>, lendo <name>, <description>, os campos de
    <ExtendedData> e a geometria (Polygon / MultiGeometry). Mais tolerante a
    KMLs nao padronizados.
    """
    from lxml import etree
    from shapely.geometry import Polygon, MultiPolygon

    with open(caminho, "rb") as f:
        conteudo = f.read()
    # Remove declaracao de encoding duplicada e faz parse tolerante
    parser = etree.XMLParser(recover=True, huge_tree=True)
    root = etree.fromstring(conteudo, parser=parser)

    def localname(tag):
        return etree.QName(tag).localname if tag is not None else ""

    def parse_coords(texto):
        pts = []
        for token in texto.replace("\n", " ").split():
            partes = token.split(",")
            if len(partes) >= 2:
                try:
                    lon, lat = float(partes[0]), float(partes[1])
                    pts.append((lon, lat))
                except ValueError:
                    continue
        return pts

    registros = []
    geometrias = []
    for pm in root.iter():
        if localname(pm.tag) != "Placemark":
            continue
        atributos = {}
        # nome e descricao
        for child in pm:
            ln = localname(child.tag)
            if ln == "name":
                atributos["name"] = (child.text or "").strip()
            elif ln == "description":
                atributos["description"] = (child.text or "").strip()

        # ExtendedData -> Data/SimpleData
        for data in pm.iter():
            ln = localname(data.tag)
            if ln in ("Data", "SimpleData"):
                nome_attr = data.get("name")
                if nome_attr:
                    valor = ""
                    if ln == "SimpleData":
                        valor = (data.text or "").strip()
                    else:  # <Data><value>...</value></Data>
                        for v in data:
                            if localname(v.tag) == "value":
                                valor = (v.text or "").strip()
                    atributos[nome_attr] = valor

        # Geometria: coleta todos os aneis de Polygon
        poligonos = []
        for poly in pm.iter():
            if localname(poly.tag) != "Polygon":
                continue
            exterior = None
            interiores = []
            for ring in poly.iter():
                if localname(ring.tag) == "outerBoundaryIs":
                    for c in ring.iter():
                        if localname(c.tag) == "coordinates":
                            exterior = parse_coords(c.text or "")
                elif localname(ring.tag) == "innerBoundaryIs":
                    for c in ring.iter():
                        if localname(c.tag) == "coordinates":
                            interiores.append(parse_coords(c.text or ""))
            if exterior and len(exterior) >= 3:
                poligonos.append(Polygon(exterior, interiores))

        if not poligonos:
            continue
        geom = poligonos[0] if len(poligonos) == 1 else MultiPolygon(poligonos)
        registros.append(atributos)
        geometrias.append(geom)

    if not geometrias:
        raise ValueError("Nenhum poligono encontrado no KML (parser fastkml/lxml).")

    gdf = gpd.GeoDataFrame(registros, geometry=geometrias, crs=f"EPSG:{EPSG_GEO}")
    return gdf


# ----------------------------- Execucao -------------------------------------
try:
    try:
        areas_gdf = ler_kml_geopandas(KML_PATH)
        print("KML lido com o driver nativo do GeoPandas.")
    except Exception as e1:
        print("Driver nativo falhou (", repr(e1), "). Tentando parser fastkml/lxml...")
        areas_gdf = ler_kml_fastkml(KML_PATH)
        print("KML lido com o parser fastkml/lxml.")
except Exception as e:
    raise RuntimeError(
        "Nao foi possivel ler o KML. Verifique se o arquivo e valido. Erro: " + repr(e)
    )

# Mantem apenas geometrias de area (Polygon / MultiPolygon)
areas_gdf = areas_gdf[areas_gdf.geometry.notna()].copy()
areas_gdf = areas_gdf[areas_gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
if areas_gdf.empty:
    raise ValueError("O KML nao contem poligonos (apenas pontos/linhas?).")

# Garante CRS WGS84
if areas_gdf.crs is None:
    areas_gdf = areas_gdf.set_crs(epsg=EPSG_GEO)
else:
    areas_gdf = areas_gdf.to_crs(epsg=EPSG_GEO)

# Corrige geometrias invalidas (auto-interseccao etc.) de forma vetorizada
invalidas = ~areas_gdf.geometry.is_valid
if invalidas.any():
    print(f"Corrigindo {int(invalidas.sum())} geometria(s) invalida(s) no KML...")
    areas_gdf.loc[invalidas, "geometry"] = areas_gdf.loc[invalidas, "geometry"].apply(make_valid)

areas_gdf = areas_gdf.reset_index(drop=True)
print(f"\nTotal de areas (poligonos) lidas: {len(areas_gdf)}")
print("Colunas/atributos disponiveis no KML:", list(areas_gdf.columns))
areas_gdf.head()


KML lido com o driver nativo do GeoPandas.
Corrigindo 16 geometria(s) invalida(s) no KML...

Total de areas (poligonos) lidas: 7707
Colunas/atributos disponiveis no KML: ['id', 'Name', 'description', 'timestamp', 'begin', 'end', 'altitudeMode', 'tessellate', 'extrude', 'visibility', 'drawOrder', 'icon', 'FAZENDA', 'ZONA', 'TALHAO', 'AREA_TOTAL', 'NOME_FAZ', 'geometry']


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,FAZENDA,ZONA,TALHAO,AREA_TOTAL,NOME_FAZ,geometry
0,0001010001,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,1.0,1.0,1.0,1.87,PÁTIO AGRÍCOLA,"MULTIPOLYGON (((-47.77229 -21.48086, -47.77126..."
1,0170010004,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,4.0,3.00,FAZENDA IPÊ,"MULTIPOLYGON (((-47.66309 -21.54266, -47.66302..."
2,0170010005,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,5.0,0.90,FAZENDA IPÊ,"MULTIPOLYGON (((-47.66196 -21.54378, -47.66197..."
3,0170010002,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,2.0,0.15,FAZENDA IPÊ,"MULTIPOLYGON (((-47.66068 -21.54376, -47.6607 ..."
4,0170010001,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,1.0,3.12,FAZENDA IPÊ,"MULTIPOLYGON (((-47.66073 -21.54355, -47.66069..."


### 3.2 — Padronização do nome e do ID da área

Define duas colunas estáveis — `area_nome` e `area_id` — usadas no restante do
notebook, escolhendo automaticamente a melhor coluna disponível para "nome" da
área. Todos os atributos originais do KML são preservados.


In [5]:
# Escolhe a melhor coluna para representar o NOME da area
CANDIDATAS_NOME = ["name", "Name", "nome", "NOME", "talhao", "Talhao",
                   "TALHAO", "setor", "bloco", "area", "AREA", "label", "title"]

col_nome = None
for c in CANDIDATAS_NOME:
    if c in areas_gdf.columns:
        # so usa se tiver pelo menos um valor nao vazio
        if areas_gdf[c].astype(str).str.strip().replace("nan", "").any():
            col_nome = c
            break

if col_nome is None:
    areas_gdf["area_nome"] = ["Area_" + str(i + 1) for i in range(len(areas_gdf))]
    print("Nenhuma coluna de nome encontrada; gerando nomes sequenciais (Area_1, Area_2, ...).")
else:
    areas_gdf["area_nome"] = (
        areas_gdf[col_nome].astype(str).str.strip().replace({"nan": "", "None": ""})
    )
    # preenche nomes vazios com sequencial
    vazios = areas_gdf["area_nome"] == ""
    areas_gdf.loc[vazios, "area_nome"] = [
        "Area_" + str(i + 1) for i in areas_gdf.index[vazios]
    ]
    print(f"Coluna de nome da area: '{col_nome}'")

# ID estavel da area (indice inteiro). Mantemos tambem como coluna.
areas_gdf["area_id"] = areas_gdf.index.astype(int)

# Lista de atributos extras do KML (tudo que nao e geometria/derivado)
ATRIBUTOS_KML = [
    c for c in areas_gdf.columns
    if c not in ("geometry", "area_id", "area_nome")
]
print("Atributos extras do KML que serao anexados aos pontos:", ATRIBUTOS_KML)
areas_gdf[["area_id", "area_nome"] + ATRIBUTOS_KML].head()


Coluna de nome da area: 'Name'
Atributos extras do KML que serao anexados aos pontos: ['id', 'Name', 'description', 'timestamp', 'begin', 'end', 'altitudeMode', 'tessellate', 'extrude', 'visibility', 'drawOrder', 'icon', 'FAZENDA', 'ZONA', 'TALHAO', 'AREA_TOTAL', 'NOME_FAZ']


,area_id,area_nome,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,FAZENDA,ZONA,TALHAO,AREA_TOTAL,NOME_FAZ
0,0,Area_1,0001010001,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,1.0,1.0,1.0,1.87,PÁTIO AGRÍCOLA
1,1,Area_2,0170010004,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,4.0,3.00,FAZENDA IPÊ
2,2,Area_3,0170010005,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,5.0,0.90,FAZENDA IPÊ
3,3,Area_4,0170010002,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,2.0,0.15,FAZENDA IPÊ
4,4,Area_5,0170010001,None,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,170.0,1.0,1.0,3.12,FAZENDA IPÊ


### 3.3 — Leitura do ZIP de telemetria → DataFrame consolidado

Abre o ZIP automaticamente, lê **todos os CSVs** (mesmo em subpastas),
consolida em um único DataFrame e valida as colunas obrigatórias.


In [6]:
# Colunas que interessam (as demais sao descartadas para poupar RAM).
COLS_TELEMETRIA = ["ceqid", "nickname", "vin", "name", "numeric_value",
                   "text_value", "uom", "event_timestamp", "lat", "lon"]
# Colunas de texto que serao convertidas para 'category' (economia de RAM).
COLS_TEXTO = ["ceqid", "nickname", "vin", "name", "text_value", "uom"]


def _detectar_sep_enc(raw_bytes):
    """Detecta encoding e separador a partir de uma amostra pequena do arquivo."""
    enc = "utf-8-sig"   # utf-8-sig remove BOM se existir
    for tentativa in ("utf-8-sig", "latin-1"):
        try:
            cabecalho = raw_bytes[:65536].decode(tentativa)
            enc = tentativa
            break
        except UnicodeDecodeError:
            enc = "latin-1"
    primeira = cabecalho.splitlines()[0] if cabecalho else ""
    # escolhe o separador mais frequente no cabecalho
    sep = max([",", ";", "\t", "|"], key=lambda s: primeira.count(s))
    return enc, sep


def ler_zip_csvs(caminho_zip):
    """Le todos os CSVs do ZIP e consolida, de forma economica em memoria.

    Estrategia robusta: le TODAS as colunas como texto (dtype=str, sem forcar
    conversao numerica na leitura, o que evita perder dados), mantem apenas as
    colunas conhecidas e converte os textos para 'category' por arquivo -> a
    economia de RAM vem da 'category', sem risco de descartar colunas/valores.
    """
    if not zipfile.is_zipfile(caminho_zip):
        raise ValueError(f"'{caminho_zip}' nao e um arquivo ZIP valido.")

    frames = []
    with zipfile.ZipFile(caminho_zip) as zf:
        nomes_csv = [
            n for n in zf.namelist()
            if n.lower().endswith(".csv") and not n.startswith("__MACOSX")
        ]
        if not nomes_csv:
            raise ValueError("O ZIP nao contem nenhum arquivo .csv.")

        print(f"{len(nomes_csv)} CSV(s) encontrado(s) no ZIP.")
        for nome in tqdm(nomes_csv, desc="Lendo CSVs"):
            raw = zf.read(nome)
            try:
                enc, sep = _detectar_sep_enc(raw)
                df_csv = pd.read_csv(
                    BytesIO(raw), sep=sep, encoding=enc,
                    dtype=str, on_bad_lines="skip", low_memory=True,
                )
            except Exception as e:
                print(f"  aviso: nao foi possivel ler '{nome}' ({e}); ignorado.")
                continue
            finally:
                del raw  # libera os bytes imediatamente
            # Normaliza nomes de colunas (remove espacos e BOM residual)
            df_csv.columns = [str(c).strip().lstrip("﻿") for c in df_csv.columns]
            if df_csv.shape[1] <= 1:
                print(f"  aviso: '{nome}' com layout inesperado; ignorado.")
                continue
            # Mantem apenas as colunas conhecidas que existem no arquivo
            manter = [c for c in df_csv.columns if c in COLS_TELEMETRIA]
            df_csv = df_csv[manter]
            # Converte textos para category (economia de RAM) por arquivo
            for c in COLS_TEXTO:
                if c in df_csv.columns:
                    df_csv[c] = df_csv[c].astype("category")
            frames.append(df_csv)

    if not frames:
        raise ValueError("Nenhum CSV pode ser lido do ZIP.")

    df = pd.concat(frames, ignore_index=True, copy=False)
    del frames
    gc.collect()
    return df


try:
    df = ler_zip_csvs(ZIP_PATH)
except Exception as e:
    raise RuntimeError("Falha ao abrir/ler o ZIP: " + repr(e))

print(f"\nTotal de linhas consolidadas: {len(df):,}".replace(",", "."))
print("Colunas encontradas:", list(df.columns))

# Validacao das colunas obrigatorias
faltando = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if faltando:
    raise ValueError(
        "Os CSVs nao possuem as colunas obrigatorias: " + ", ".join(faltando) +
        ".\nColunas presentes: " + ", ".join(df.columns)
    )

# Garante a existencia das colunas opcionais esperadas (preenche vazio se faltar)
for c in ["nickname", "vin", "name", "numeric_value", "text_value", "uom"]:
    if c not in df.columns:
        df[c] = np.nan

df.head()


1 CSV(s) encontrado(s) no ZIP.


Lendo CSVs:   0%|          | 0/1 [00:00<?, ?it/s]


Total de linhas consolidadas: 3.339.999
Colunas encontradas: ['ceqid', 'nickname', 'vin', 'name', 'numeric_value', 'text_value', 'uom', 'event_timestamp', 'lat', 'lon']


,ceqid,nickname,vin,name,numeric_value,text_value,uom,event_timestamp,lat,lon
0,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_ALT,663.44,NaN,M,2026-04-01T01:03:09-03:00,-21.532349333333332,-47.7330305
1,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_CURRENT,0.0,NaN,NaN,2026-04-01T01:03:09-03:00,-21.532349333333332,-47.7330305
2,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_SPEED,0.0,NaN,KM/H,2026-04-01T01:03:09-03:00,-21.532349333333332,-47.7330305
3,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_PDOP,4.3,NaN,NaN,2026-04-01T01:03:09-03:00,-21.532349333333332,-47.7330305
4,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_SAT,5.0,NaN,NaN,2026-04-01T01:03:09-03:00,-21.532349333333332,-47.7330305


### 3.4 — Validação e limpeza dos registros

Converte `lat`, `lon` e `event_timestamp` para os tipos corretos e **remove
registros inválidos**: coordenadas ausentes/fora de faixa, timestamps que não
podem ser interpretados, e pontos em (0, 0).


In [7]:
n_inicial = len(df)

# Converte lat/lon para numerico (tolera virgula decimal pt-BR; invalidos -> NaN)
df["lat"] = para_numero(df["lat"])
df["lon"] = para_numero(df["lon"])

# Converte o timestamp. utc=True garante consistencia de fuso.
df["event_timestamp"] = pd.to_datetime(
    df["event_timestamp"], errors="coerce", utc=True
)

# Mascara de registros validos (operacao vetorizada)
valido = (
    df["lat"].notna() & df["lon"].notna() &
    df["event_timestamp"].notna() &
    df["lat"].between(-90, 90) & df["lon"].between(-180, 180) &
    ~((df["lat"] == 0) & (df["lon"] == 0))            # descarta (0,0)
)

removidos = int((~valido).sum())
df = df[valido].reset_index(drop=True)   # filtra (sem .copy() extra p/ poupar RAM)
del valido
gc.collect()

def _aplica_em_texto(serie, func):
    """Aplica 'func' a uma coluna de texto de forma economica em RAM.

    Se a coluna for 'category', transforma apenas as CATEGORIAS (poucas), sem
    expandir para milhoes de strings 'object'. Caso contrario usa mapa de unicos.
    """
    if isinstance(serie.dtype, pd.CategoricalDtype):
        novas = {c: func(c) for c in serie.cat.categories}
        # rename_categories exige categorias unicas; se houver colisao, recategoriza
        if len(set(novas.values())) == len(novas):
            return serie.cat.rename_categories(novas)
        return serie.map(novas).astype("category")
    unicos = serie.dropna().unique()
    mapa = {v: func(v) for v in unicos}
    return serie.map(mapa).where(serie.notna(), serie)

# Normaliza ceqid (mantendo como category para economizar memoria)
df["ceqid"] = _aplica_em_texto(df["ceqid"], lambda x: str(x).strip())

# Corrige acentuacao quebrada (mojibake) nas colunas de texto
# Ex.: "Status da orientaÃ§Ã£o automÃ¡tica" -> "Status da orientação automática"
for col in ["name", "text_value", "nickname"]:
    if col in df.columns:
        df[col] = _aplica_em_texto(df[col], corrigir_mojibake)

print(f"Registros iniciais : {n_inicial:,}".replace(",", "."))
print(f"Registros removidos: {removidos:,}".replace(",", "."))
print(f"Registros validos  : {len(df):,}".replace(",", "."))
if df.empty:
    raise ValueError("Nenhum registro valido restante apos a limpeza.")
df.head()


Registros iniciais : 3.339.999
Registros removidos: 2.679
Registros validos  : 3.337.320


,ceqid,nickname,vin,name,numeric_value,text_value,uom,event_timestamp,lat,lon
0,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_ALT,663.44,NaN,M,2026-04-01 04:03:09+00:00,-21.532349,-47.73303
1,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_CURRENT,0.0,NaN,NaN,2026-04-01 04:03:09+00:00,-21.532349,-47.73303
2,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_SPEED,0.0,NaN,KM/H,2026-04-01 04:03:09+00:00,-21.532349,-47.73303
3,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_PDOP,4.3,NaN,NaN,2026-04-01 04:03:09+00:00,-21.532349,-47.73303
4,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,GPS_SAT,5.0,NaN,NaN,2026-04-01 04:03:09+00:00,-21.532349,-47.73303


### 3.5 — Reconstrução dos estados de operação (sinais da coluna `name`)

A telemetria está em **formato longo**: cada linha é uma leitura de **um** sinal
(coluna `name`), com o valor em `numeric_value` ou `text_value`. Para calcular os
tempos por estado, reconstruímos uma tabela com **uma linha por amostra**
(`ceqid` + `event_timestamp` + posição) e, nela, o valor de cada sinal de interesse:

- **Carga do motor** (`numeric_value`, %) — sinal cujo `name` contém *carga* e *motor*.
- **Status do elevador** (`text_value`) — sinal cujo `name` contém *elevador*.
- **Status da orientação automática / piloto** (`text_value`) — `name` contém *orientação automática*.

Com base nesses sinais marcamos dois estados:

| Estado | Regra |
|---|---|
| **Tempo efetivo** | carga do motor **≥ 30%** **E** status do elevador = `forward` |
| **Piloto ligado** | Status da orientação automática = `on` (ligado) |

Como esses sinais costumam ser registrados **apenas quando mudam**, aplicamos
*forward-fill* (último valor conhecido) por equipamento ao longo do tempo, para
que o estado persista até a próxima mudança.


In [8]:
# ------------------------------------------------------------------
# Identifica os SINAIS trabalhando no nivel das CATEGORIAS do 'name'
# (ha poucos nomes distintos) -> nao cria colunas gigantes na memoria.
# ------------------------------------------------------------------
nomes_distintos = pd.unique(df["name"].astype(str).dropna())
norm_por_nome = {n: remover_acentos(n) for n in nomes_distintos}

def _nomes_que_contem(palavras):
    """Retorna os nomes de sinal cujo texto normalizado contem TODAS as palavras."""
    pals = [remover_acentos(p) for p in palavras]
    return [n for n, norm in norm_por_nome.items() if all(p in norm for p in pals)]

nomes_carga = _nomes_que_contem(SINAL_CARGA_MOTOR)
nomes_elev = _nomes_que_contem(SINAL_STATUS_ELEVADOR)
nomes_piloto = _nomes_que_contem(SINAL_PILOTO)

# Mascaras booleanas por pertencimento ao conjunto (barato em RAM)
name_str = df["name"].astype(str)
mask_carga = name_str.isin(nomes_carga)
mask_elev = name_str.isin(nomes_elev)
mask_piloto = name_str.isin(nomes_piloto)
del name_str

print("Sinais identificados na coluna 'name':")
print("  Carga do motor       :", nomes_carga[:5] or ">>> NENHUM! ajuste SINAL_CARGA_MOTOR <<<")
print("  Status do elevador   :", nomes_elev[:5] or ">>> NENHUM! ajuste SINAL_STATUS_ELEVADOR <<<")
print("  Orientacao automatica:", nomes_piloto[:5] or ">>> NENHUM! ajuste SINAL_PILOTO <<<")
if not (nomes_carga and nomes_elev and nomes_piloto):
    print("\n  ATENCAO: algum sinal NAO foi encontrado -> as horas efetivas/piloto")
    print("  correspondentes virao ZERADAS ate ajustar as palavras-chave no topo.")
# Sempre lista os nomes de sinais presentes, para facilitar a configuracao.
print("\n  >> Nomes de sinais presentes na coluna 'name' (top 30 por frequencia):")
for nm, qt in df["name"].astype(str).value_counts().head(30).items():
    print(f"     - {nm}  ({qt})")

# ------------------------------------------------------------------
# nickname/vin sao propriedades do EQUIPAMENTO: montamos uma tabela
# ceqid -> nickname/vin com o 1o valor NAO-VAZIO (via groupby.first(),
# que ignora NaN). Depois fazemos MERGE em samples (robusto a dtypes).
# ------------------------------------------------------------------
ident = df[["ceqid"]].drop_duplicates().copy()
for coluna in ["nickname", "vin"]:
    if coluna in df.columns:
        tmp = df[["ceqid", coluna]].copy()
        # texto limpo; vazios/placeholder viram NaN para o first() ignora-los
        vals = tmp[coluna].astype(str).str.strip()
        vals = vals.where(~vals.isin(["", "nan", "None", "NaN", "<NA>"]))
        tmp[coluna] = vals
        primeiro = (
            tmp.dropna(subset=[coluna])
            .groupby("ceqid", observed=True)[coluna].first()
            .reset_index()
        )
        ident = ident.merge(primeiro, on="ceqid", how="left")
    else:
        ident[coluna] = np.nan
# ceqid como texto para o merge casar com samples de forma consistente
ident["ceqid"] = ident["ceqid"].astype(str)
print("Equipamentos (ceqid) com nickname preenchido:",
      int(ident["nickname"].notna().sum()), "de", len(ident))

# ------------------------------------------------------------------
# Monta a tabela de AMOSTRAS: uma linha por (ceqid, event_timestamp)
# ------------------------------------------------------------------
chave = ["ceqid", "event_timestamp"]

# Valores representativos por amostra (apenas a posicao; nickname/vin vem do
# lookup por equipamento mais abaixo). groupby sem pre-sort evita copiar o df.
samples = (
    df.groupby(chave, as_index=False, observed=True, sort=False)
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
    )
)

def _valor_sinal(mask, coluna):
    """Valor do sinal (coluna) por amostra (ceqid,event_timestamp). Pega o 1o por chave."""
    if not mask.any():
        return pd.Series(index=pd.MultiIndex.from_arrays([[], []], names=chave), dtype=object)
    sub = df.loc[mask, chave + [coluna]].dropna(subset=[coluna])
    if sub.empty:
        return pd.Series(index=pd.MultiIndex.from_arrays([[], []], names=chave), dtype=object)
    return sub.groupby(chave, observed=True)[coluna].first()

# Extrai o valor de cada sinal por amostra e junta na tabela de amostras.
# A carga usa 'para_numero' (tolera virgula decimal) -> nao zera o tempo efetivo.
carga_serie = para_numero(_valor_sinal(mask_carga, "numeric_value"))
elev_serie = _valor_sinal(mask_elev, "text_value").astype("object")
piloto_serie = _valor_sinal(mask_piloto, "text_value").astype("object")
del mask_carga, mask_elev, mask_piloto

samples = samples.set_index(chave)
samples["carga_motor"] = carga_serie.astype("float32")
samples["status_elevador"] = elev_serie
samples["status_piloto"] = piloto_serie
samples = samples.reset_index()
del carga_serie, elev_serie, piloto_serie

# nickname/vin por equipamento: merge com a tabela 'ident' (1o nao-vazio por ceqid)
samples["ceqid"] = samples["ceqid"].astype(str)
samples = samples.merge(ident, on="ceqid", how="left")
samples["nickname"] = samples["nickname"].fillna("")
samples["vin"] = samples["vin"].fillna("")

# A partir daqui nao precisamos mais do DataFrame longo (telemetria bruta).
del df, ident
gc.collect()

# Ordena por equipamento e tempo e aplica forward-fill por equipamento
# (o estado persiste ate a proxima leitura que o altere)
samples = samples.sort_values(chave).reset_index(drop=True)
for col in ["carga_motor", "status_elevador", "status_piloto"]:
    samples[col] = samples.groupby("ceqid", observed=True)[col].ffill()

# ------------------------------------------------------------------
# Marca os estados de operacao (flags booleanas vetorizadas)
# ------------------------------------------------------------------
_map_elev = {v: remover_acentos(v) for v in samples["status_elevador"].dropna().unique()}
_map_pil = {v: remover_acentos(v) for v in samples["status_piloto"].dropna().unique()}
elev_norm = samples["status_elevador"].map(_map_elev).fillna("")
piloto_norm = samples["status_piloto"].map(_map_pil).fillna("")
valores_elev_ok = [remover_acentos(v) for v in VALOR_ELEVADOR_EFETIVO]
valores_on = [remover_acentos(v) for v in VALORES_LIGADO]

samples["elevador_forward"] = elev_norm.isin(valores_elev_ok)
samples["piloto_ligado"] = piloto_norm.isin(valores_on)
samples["motor_acima_limiar"] = samples["carga_motor"] >= LIMIAR_CARGA_MOTOR
# Tempo efetivo = carga >= LIMIAR_CARGA_MOTOR E elevador "forward"
samples["efetivo"] = samples["motor_acima_limiar"] & samples["elevador_forward"]

# Substitui 'df' pela tabela de amostras no restante do fluxo
df = samples

# ------------------------------------------------------------------
# DIAGNOSTICO DOS VALORES: mostra o que foi lido em cada sinal, para
# confirmar se 'forward'/'on'/carga estao casando (evita horas zeradas).
# ------------------------------------------------------------------
print("\n--- Diagnostico dos valores por sinal ---")
carga_ok = int(df["carga_motor"].notna().sum())
if carga_ok:
    print(f"Carga do motor: {carga_ok:,} amostras com valor | ".replace(",", ".") +
          f"min={df['carga_motor'].min():.1f} max={df['carga_motor'].max():.1f} " +
          f"| >= {LIMIAR_CARGA_MOTOR:.0f}%: {int(df['motor_acima_limiar'].sum()):,}".replace(",", "."))
else:
    print("Carga do motor: NENHUM valor numerico lido (verifique o sinal/numeric_value).")
vals_elev = df["status_elevador"].dropna().astype(str).value_counts().head(8)
vals_pil = df["status_piloto"].dropna().astype(str).value_counts().head(8)
print("Valores do status do elevador (esperado casar com "
      f"{VALOR_ELEVADOR_EFETIVO}): {dict(vals_elev)}")
print(f"Valores do status do piloto (esperado casar com 'on'/ligado): {dict(vals_pil)}")

print(f"\nTotal de amostras (ceqid + timestamp): {len(df):,}".replace(",", "."))
print(f"  Amostras com piloto ligado             : {int(df['piloto_ligado'].sum()):,}".replace(",", "."))
print(f"  Amostras em tempo efetivo (>={LIMIAR_CARGA_MOTOR:.0f}% + fwd): {int(df['efetivo'].sum()):,}".replace(",", "."))
df[["ceqid", "event_timestamp", "carga_motor", "status_elevador",
    "status_piloto", "efetivo", "piloto_ligado"]].head(10)


Sinais identificados na coluna 'name':
  Carga do motor       : ['Carga do Motor']
  Status do elevador   : ['Status do Elevador ', 'Horas do elevador']
  Orientacao automatica: ['Status da orientação automática']

  >> Nomes de sinais presentes na coluna 'name' (top 30 por frequencia):
     - GSS_FIELD  (84398)
     - GPS_ALT  (78987)
     - GPS_CURRENT  (78987)
     - GPS_SPEED  (78987)
     - GPS_SAT  (78987)
     - GPS_PDOP  (78987)
     - GPS_DIR  (78987)
     - STATUS_DEVICE  (78987)
     - NETWORK_RSSI  (78987)
     - GPS_FIX  (78987)
     - NETWORK_OPERATOR_NAME  (78987)
     - NETWORK_STATUS  (78987)
     - NETWORK_CONNECTION  (78987)
     - STATUS_DUTY_CODE  (78893)
     - STATUS_DUTY  (78893)
     - STATUS_DUTY_DESCRIPTION  (78893)
     - Taxa de combustível do motor  (61568)
     - Rotação do Motor Baixa  (61510)
     - Status do operador sentado  (61510)
     - Pressão hidráulica do cortador de base  (61487)
     - Altura Ct. Base  (61480)
     - Sentido do corte de base  

,ceqid,event_timestamp,carga_motor,status_elevador,status_piloto,efetivo,piloto_ligado
0,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:09+00:00,NaN,NaN,NaN,False,False
1,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:14+00:00,NaN,NaN,NaN,False,False
2,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:17+00:00,NaN,NaN,NaN,False,False
3,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:18+00:00,NaN,NaN,NaN,False,False
4,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:21+00:00,NaN,NaN,NaN,False,False
5,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:04:17+00:00,11.0,Neutral,Off,False,False
6,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:05:17+00:00,11.0,Neutral,Off,False,False
7,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:06:18+00:00,11.0,Neutral,Off,False,False
8,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:07:18+00:00,10.0,Neutral,Off,False,False
9,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:08:18+00:00,10.0,Neutral,Off,False,False


## 🌍 Etapa 4 — Conversão espacial (lat/lon → pontos geográficos)

Cria um `GeoDataFrame` de pontos usando **EPSG:4326 (WGS84)**. A construção da
geometria é feita de forma **vetorizada** com `gpd.points_from_xy`, adequada a
grandes volumes. A partir daqui trabalhamos com a **tabela de amostras** (uma
linha por posição/instante) construída na Etapa 3.5.


In [9]:
# Criacao vetorizada da geometria de pontos (rapido para centenas de milhares)
pontos_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=f"EPSG:{EPSG_GEO}",
)
print(f"GeoDataFrame de pontos criado: {len(pontos_gdf):,} pontos.".replace(",", "."))
print("CRS dos pontos:", pontos_gdf.crs)
pontos_gdf[["ceqid", "lat", "lon", "event_timestamp", "geometry"]].head()


GeoDataFrame de pontos criado: 78.987 pontos.
CRS dos pontos: EPSG:4326


,ceqid,lat,lon,event_timestamp,geometry
0,76d8b989-cca7-215f-9b84-afb26c66a151,-21.532349,-47.73303,2026-04-01 04:03:09+00:00,POINT (-47.73303 -21.53235)
1,76d8b989-cca7-215f-9b84-afb26c66a151,-21.532349,-47.73303,2026-04-01 04:03:14+00:00,POINT (-47.73303 -21.53235)
2,76d8b989-cca7-215f-9b84-afb26c66a151,-21.532349,-47.73303,2026-04-01 04:03:17+00:00,POINT (-47.73303 -21.53235)
3,76d8b989-cca7-215f-9b84-afb26c66a151,-21.532349,-47.73303,2026-04-01 04:03:18+00:00,POINT (-47.73303 -21.53235)
4,76d8b989-cca7-215f-9b84-afb26c66a151,-21.532349,-47.73303,2026-04-01 04:03:21+00:00,POINT (-47.73303 -21.53235)


## 🔗 Etapa 5 — Relacionamento espacial (Point-in-Polygon)

Executa o **Spatial Join** do GeoPandas (`sjoin`, predicado `within`), que usa
**índice espacial (R-tree)** automaticamente. Para cada ponto identificamos o
**nome** e o **ID** da área correspondente — o suficiente para o resumo por
equipamento — sem loops linha a linha.

Pontos que caem em mais de um polígono (áreas sobrepostas) são resolvidos
mantendo a primeira correspondência para não duplicar registros. A geometria é
descartada logo após o join para poupar memória.


In [10]:
# So precisamos do ID e do NOME da area por ponto (economico em memoria).
areas_join = areas_gdf[["area_id", "area_nome", "geometry"]].copy()

# Spatial join vetorizado com indice espacial (predicate='within')
print("Executando Spatial Join (Point-in-Polygon) com indice espacial...")
juntado = gpd.sjoin(pontos_gdf, areas_join, how="left", predicate="within")
del pontos_gdf, areas_join
gc.collect()

# Em caso de sobreposicao, um ponto pode duplicar; mantem a 1a area encontrada
juntado = juntado[~juntado.index.duplicated(keep="first")]
# Descarta geometria e coluna auxiliar do join (nao usadas no resumo)
juntado = pd.DataFrame(
    juntado.drop(columns=[c for c in ["index_right", "geometry"]
                          if c in juntado.columns])
)
gc.collect()

# Marca pontos sem area
juntado["area_nome"] = juntado["area_nome"].fillna("SEM AREA")
sem_area = int((juntado["area_nome"] == "SEM AREA").sum())

print(f"Pontos relacionados a alguma area: {len(juntado) - sem_area:,}".replace(",", "."))
print(f"Pontos SEM area associada        : {sem_area:,}".replace(",", "."))
juntado[["ceqid", "event_timestamp", "area_nome", "area_id"]].head()


Executando Spatial Join (Point-in-Polygon) com indice espacial...
Pontos relacionados a alguma area: 62.146
Pontos SEM area associada        : 16.841


,ceqid,event_timestamp,area_nome,area_id
0,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:09+00:00,Area_6267,6266.0
1,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:14+00:00,Area_6267,6266.0
2,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:17+00:00,Area_6267,6266.0
3,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:18+00:00,Area_6267,6266.0
4,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:21+00:00,Area_6267,6266.0


## ⏱️ Etapa 6 — Cálculo do tempo trabalhado

Ordena por `ceqid` e `event_timestamp` e calcula a diferença de tempo entre
pontos consecutivos **do mesmo equipamento** (vetorizado com `groupby().diff()`).

**Regras aplicadas:**
- Intervalos **negativos** são ignorados (timestamps fora de ordem).
- Intervalos **maiores que 10 minutos** são ignorados (equipamento parado/desligado).
- O tempo é atribuído ao ponto **atual** (e à sua área), representando o trabalho
  realizado desde o ponto anterior.

São criadas as colunas `tempo_segundos`, `tempo_minutos` e `tempo_horas`.

Além do tempo total, o mesmo intervalo é classificado por **estado de operação**
(definido na Etapa 3.5), gerando:

- `tempo_efetivo_segundos` / `tempo_efetivo_horas` — **tempo efetivo** (carga ≥ 30% e elevador `forward`).
- `tempo_piloto_segundos` / `tempo_piloto_horas` — **tempo com piloto ligado** (orientação automática `on`).


In [11]:
# Ordena por equipamento e tempo (essencial para o diff temporal)
juntado = juntado.sort_values(["ceqid", "event_timestamp"]).reset_index(drop=True)

# Diferenca de tempo (em segundos) em relacao ao ponto anterior do MESMO ceqid
delta = juntado.groupby("ceqid")["event_timestamp"].diff().dt.total_seconds()

# Aplica as regras: descarta negativos e gaps > 10 min; NaN (1o ponto) -> 0
delta = delta.where((delta >= 0) & (delta <= MAX_GAP_SECONDS))
juntado["tempo_segundos"] = delta.fillna(0.0)
juntado["tempo_minutos"] = juntado["tempo_segundos"] / 60.0
juntado["tempo_horas"] = juntado["tempo_segundos"] / 3600.0

# Tempo por ESTADO de operacao: o intervalo so conta para o estado se a
# amostra atual estiver naquele estado (efetivo / piloto ligado).
juntado["tempo_efetivo_segundos"] = np.where(
    juntado["efetivo"], juntado["tempo_segundos"], 0.0
)
juntado["tempo_efetivo_horas"] = juntado["tempo_efetivo_segundos"] / 3600.0

juntado["tempo_piloto_segundos"] = np.where(
    juntado["piloto_ligado"], juntado["tempo_segundos"], 0.0
)
juntado["tempo_piloto_horas"] = juntado["tempo_piloto_segundos"] / 3600.0

total_horas = juntado["tempo_horas"].sum()
total_efetivo = juntado["tempo_efetivo_horas"].sum()
total_piloto = juntado["tempo_piloto_horas"].sum()
print(f"Tempo total valido    : {total_horas:,.2f} h".replace(",", "."))
print(f"Tempo EFETIVO (>={LIMIAR_CARGA_MOTOR:.0f}% + elevador forward): {total_efetivo:,.2f} h".replace(",", "."))
print(f"Tempo PILOTO (orientacao automatica on) : {total_piloto:,.2f} h".replace(",", "."))
juntado[["ceqid", "event_timestamp", "area_nome", "carga_motor",
         "status_elevador", "status_piloto", "tempo_horas",
         "tempo_efetivo_horas", "tempo_piloto_horas"]].head(10)


Tempo total valido    : 1.077.94 h
Tempo EFETIVO (>=30% + elevador forward): 331.76 h
Tempo PILOTO (orientacao automatica on) : 107.84 h


,ceqid,event_timestamp,area_nome,carga_motor,status_elevador,status_piloto,tempo_horas,tempo_efetivo_horas,tempo_piloto_horas
0,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:09+00:00,Area_6267,NaN,NaN,NaN,0.000000,0.0,0.0
1,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:14+00:00,Area_6267,NaN,NaN,NaN,0.001389,0.0,0.0
2,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:17+00:00,Area_6267,NaN,NaN,NaN,0.000833,0.0,0.0
3,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:18+00:00,Area_6267,NaN,NaN,NaN,0.000278,0.0,0.0
4,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:03:21+00:00,Area_6267,NaN,NaN,NaN,0.000833,0.0,0.0
5,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:04:17+00:00,Area_6267,11.0,Neutral,Off,0.015556,0.0,0.0
6,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:05:17+00:00,Area_6267,11.0,Neutral,Off,0.016667,0.0,0.0
7,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:06:18+00:00,Area_6267,11.0,Neutral,Off,0.016944,0.0,0.0
8,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:07:18+00:00,Area_6267,10.0,Neutral,Off,0.016667,0.0,0.0
9,76d8b989-cca7-215f-9b84-afb26c66a151,2026-04-01 04:08:18+00:00,Area_6267,10.0,Neutral,Off,0.016667,0.0,0.0


## 🚜 Etapa 7 — Resumo por equipamento

Agrega por **equipamento, área e data**, trazendo `data`, `ceqid`, `nickname`,
`vin`, área, horas trabalhadas, **horas efetivas**, **horas com piloto ligado** e
quantidade de pontos.

A coluna **`data`** (dia do registro) é derivada do `event_timestamp` convertido
para o fuso **local** (`TIMEZONE_LOCAL`, padrão horário de Brasília), para
permitir **filtrar por data** na planilha. Assim cada linha representa o trabalho
de um equipamento, em uma área, em um dia (no horário local).


In [12]:
def primeiro_nao_vazio(serie):
    """Retorna o primeiro valor nao nulo/nao vazio de uma serie (para nickname/vin)."""
    s = serie.dropna().astype(str)
    s = s[s.str.strip() != ""]
    return s.iloc[0] if len(s) else ""

# Converte o timestamp (UTC) para o fuso local antes de extrair a data.
# Assim o "dia" respeita o horario de Brasilia (e nao o UTC).
try:
    ts_local = juntado["event_timestamp"].dt.tz_convert(TIMEZONE_LOCAL)
except Exception as e:
    print(f"Aviso: nao foi possivel aplicar o fuso '{TIMEZONE_LOCAL}' ({e}); usando UTC.")
    ts_local = juntado["event_timestamp"].dt.tz_convert("UTC")

# Coluna de data (apenas o dia, no horario local) para permitir filtro por data.
juntado["data"] = ts_local.dt.date

resumo_equip = (
    juntado.groupby(["data", "ceqid", "area_nome"], dropna=False)
    .agg(
        nickname=("nickname", primeiro_nao_vazio),
        vin=("vin", primeiro_nao_vazio),
        horas_trabalhadas=("tempo_horas", "sum"),
        horas_efetivas=("tempo_efetivo_horas", "sum"),
        horas_piloto=("tempo_piloto_horas", "sum"),
        qtd_pontos=("event_timestamp", "size"),
    )
    .reset_index()
)

# Reordena conforme a entrega esperada (data em primeiro para facilitar o filtro)
resumo_equip = resumo_equip[[
    "data", "ceqid", "nickname", "vin", "area_nome", "horas_trabalhadas",
    "horas_efetivas", "horas_piloto", "qtd_pontos",
]].sort_values(
    ["data", "ceqid", "horas_trabalhadas"], ascending=[True, True, False]
).reset_index(drop=True)

for c in ["horas_trabalhadas", "horas_efetivas", "horas_piloto"]:
    resumo_equip[c] = resumo_equip[c].round(2)

print(f"Resumo gerado para {resumo_equip['ceqid'].nunique()} equipamento(s).")
resumo_equip.head(20)


Resumo gerado para 1 equipamento(s).


,data,ceqid,nickname,vin,area_nome,horas_trabalhadas,horas_efetivas,horas_piloto,qtd_pontos
0,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_6267,2.93,0.00,0.0,256
1,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5860,0.43,0.00,0.0,60
2,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,SEM AREA,0.27,0.00,0.0,26
3,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5863,0.19,0.00,0.0,22
4,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5861,0.10,0.00,0.0,6
5,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_919,0.09,0.00,0.0,12
6,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5862,0.08,0.00,0.0,10
7,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_7066,0.08,0.00,0.0,6
8,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5872,0.03,0.00,0.0,2
9,2026-04-01,76d8b989-cca7-215f-9b84-afb26c66a151,PRCY9900HPPA06701,PRCY9900HPPA06701,Area_5718,0.03,0.00,0.0,8


## 💾 Etapa 8 — Relatório final + download

Exporta **apenas o resumo por equipamento** (`resumo_por_equipamento.xlsx`) e o
disponibiliza para download. Os timestamps/datas usam o fuso local
(`TIMEZONE_LOCAL`, padrão horário de Brasília).


In [13]:
def _df_excel_safe(df_in):
    """Converte colunas datetime para texto no fuso LOCAL (Excel-friendly).

    O Excel nao suporta datetimes com timezone; convertemos para o horario local
    (TIMEZONE_LOCAL, padrao Brasilia) e gravamos como texto 'AAAA-MM-DD HH:MM:SS'.
    """
    out = df_in.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            try:
                serie = out[col].dt.tz_convert(TIMEZONE_LOCAL)
            except Exception:
                try:
                    serie = out[col].dt.tz_convert("UTC")
                except (TypeError, AttributeError):
                    out[col] = out[col].astype(str)
                    continue
            out[col] = serie.dt.strftime("%Y-%m-%d %H:%M:%S")
    return out

ARQ_EQUIP = "resumo_por_equipamento.xlsx"

print("Gerando 'resumo_por_equipamento.xlsx'...")
_df_excel_safe(resumo_equip).to_excel(ARQ_EQUIP, index=False, engine="openpyxl")
print(f"Arquivo gerado: {ARQ_EQUIP} ({os.path.getsize(ARQ_EQUIP)/1024:.1f} KB)")

# Disponibiliza para download no Colab
if EM_COLAB:
    files.download(ARQ_EQUIP)


Gerando 'resumo_por_equipamento.xlsx'...
Arquivo gerado: resumo_por_equipamento.xlsx (24.1 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📈 Etapa 9 — Métricas finais

Resumo geral do processamento.


In [14]:
total_pontos = len(juntado)
total_areas = juntado.loc[juntado["area_nome"] != "SEM AREA", "area_nome"].nunique()
total_equip = juntado["ceqid"].nunique()
total_horas = juntado["tempo_horas"].sum()
total_efetivo = juntado["tempo_efetivo_horas"].sum()
total_piloto = juntado["tempo_piloto_horas"].sum()
pct_efetivo = (100 * total_efetivo / total_horas) if total_horas > 0 else 0.0
pct_piloto = (100 * total_piloto / total_horas) if total_horas > 0 else 0.0
pontos_sem_area = int((juntado["area_nome"] == "SEM AREA").sum())

print("=" * 56)
print("            METRICAS FINAIS DO PROCESSAMENTO")
print("=" * 56)
print(f"Total de pontos processados        : {total_pontos:,}".replace(",", "."))
print(f"Total de areas encontradas         : {total_areas:,}".replace(",", "."))
print(f"Total de equipamentos              : {total_equip:,}".replace(",", "."))
print(f"Total de horas calculadas          : {total_horas:,.2f} h".replace(",", "."))
print(f"Horas EFETIVAS (>={LIMIAR_CARGA_MOTOR:.0f}% + forward): {total_efetivo:,.2f} h ({pct_efetivo:.1f}%)".replace(",", "."))
print(f"Horas com PILOTO ligado            : {total_piloto:,.2f} h ({pct_piloto:.1f}%)".replace(",", "."))
print(f"Pontos SEM area associada          : {pontos_sem_area:,}".replace(",", "."))
print("=" * 56)

# Tabela-resumo das metricas
pd.DataFrame({
    "Metrica": [
        "Total de pontos processados",
        "Total de areas encontradas",
        "Total de equipamentos",
        "Total de horas calculadas",
        f"Horas efetivas (>={LIMIAR_CARGA_MOTOR:.0f}% + elevador forward)",
        "Horas com piloto ligado (orientacao automatica)",
        "Pontos sem area associada",
    ],
    "Valor": [
        total_pontos, total_areas, total_equip,
        round(total_horas, 2), round(total_efetivo, 2),
        round(total_piloto, 2), pontos_sem_area,
    ],
})


            METRICAS FINAIS DO PROCESSAMENTO
Total de pontos processados        : 78.987
Total de areas encontradas         : 145
Total de equipamentos              : 1
Total de horas calculadas          : 1.077.94 h
Horas EFETIVAS (>=30% + forward): 331.76 h (30.8%)
Horas com PILOTO ligado            : 107.84 h (10.0%)
Pontos SEM area associada          : 16.841


,Metrica,Valor
0,Total de pontos processados,78987.00
1,Total de areas encontradas,145.00
2,Total de equipamentos,1.00
3,Total de horas calculadas,1077.94
4,Horas efetivas (>=30% + elevador forward),331.76
5,Horas com piloto ligado (orientacao automatica),107.84
6,Pontos sem area associada,16841.00


---

✅ **Processamento concluído!** O arquivo **`resumo_por_equipamento.xlsx`** foi
gerado e baixado automaticamente. Ele traz, por **data / equipamento / área**, as
horas trabalhadas, as **horas efetivas** (carga ≥ 30% + elevador `forward`) e as
**horas com piloto ligado** (orientação automática `on`) — permitindo filtrar por
data. Se o download não iniciar, encontre o arquivo no painel de **Arquivos**
(ícone de pasta) à esquerda no Colab.
